# Phase 4 — Benchmark Notebook

Loads the eval JSONs generated by Step 6 and produces visualizations referenced in `phase4_report.md`.

## Sections
1. Dataset stats (per-spec scenarios, turns, audio duration distributions)
2. STT WER/DER per-spec (clean vs noisy)
3. SOAP metrics per-spec (schema/dx/dx_sem/icd/cc/plan)
4. Cloud agent per-spec (recall/precision/nfz)
5. E2E latency budget
6. Confusion-like analyses

## Usage
```bash
uv pip install jupyter matplotlib seaborn pandas
uv run jupyter lab research_and_testing/notebooks/phase4_benchmark.ipynb
```


In [ ]:
import collections
import json
import statistics
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path('..').resolve().parent
STT = json.loads((ROOT / 'research_and_testing/results/stt_eval_20260523_192721.json').read_text())
SLM = json.loads((ROOT / 'research_and_testing/results/slm_eval_20260523_234832.json').read_text())
AGENT = json.loads((ROOT / 'research_and_testing/results/agent_eval_20260523_221646.json').read_text())

scenarios = [json.loads(p.read_text('utf-8')) for p in sorted((ROOT / 'dataset_pipeline/output/scenarios').glob('*.json'))]
sid_to_scen = {s['id'][:8]: s for s in scenarios}
print(f'Loaded {len(scenarios)} scenarios')
print(f'STT results: {len(STT["results"])}, SLM rows: {len(SLM["rows"])}, agent rows: {len(AGENT["rows"])}')

## 1. Dataset stats


In [ ]:
by_spec = collections.Counter(s['specialization'] for s in scenarios)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(by_spec.keys(), by_spec.values(), color='#4f8fbf')
ax.set_ylabel('# scenariuszy')
ax.set_title('Rozkład scenariuszy per specjalizacja')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
turns = [len(s['turns']) for s in scenarios]
ages = [s['patient_age'] for s in scenarios]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(turns, bins=20, color='#4f8fbf', edgecolor='white')
axes[0].set_xlabel('# tur dialogu')
axes[0].set_title(f'Długość dialogu (mean {statistics.mean(turns):.1f}, range {min(turns)}-{max(turns)})')
axes[1].hist(ages, bins=20, color='#5fbf8f', edgecolor='white')
axes[1].set_xlabel('wiek pacjenta')
axes[1].set_title(f'Wiek (mean {statistics.mean(ages):.0f}, mediana {statistics.median(ages):.0f})')
plt.tight_layout()
plt.show()

## 2. STT WER/DER per-spec


In [ ]:
stt_by_spec = collections.defaultdict(list)
for r in STT['results']:
    sid = r['scenario_id'][:8]
    scen = sid_to_scen.get(sid)
    if scen:
        stt_by_spec[scen['specialization']].append(r)

stt_df = pd.DataFrame([{
    'spec': spec,
    'clean_WER': statistics.mean(r['clean_wer'] for r in rs) * 100,
    'noisy_WER': statistics.mean(r['noisy_wer'] for r in rs) * 100,
    'clean_DER': statistics.mean(r['clean_der'] for r in rs) * 100,
    'noisy_DER': statistics.mean(r['noisy_der'] for r in rs) * 100,
} for spec, rs in stt_by_spec.items()])
stt_df = stt_df.sort_values('spec')
stt_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = range(len(stt_df))
w = 0.35

axes[0].bar([i - w/2 for i in x], stt_df['clean_WER'], w, label='clean', color='#4f8fbf')
axes[0].bar([i + w/2 for i in x], stt_df['noisy_WER'], w, label='noisy', color='#bf8f4f')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(stt_df['spec'], rotation=30, ha='right')
axes[0].set_ylabel('WER (%)')
axes[0].set_title('WER per specjalizacja')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar([i - w/2 for i in x], stt_df['clean_DER'], w, label='clean', color='#4f8fbf')
axes[1].bar([i + w/2 for i in x], stt_df['noisy_DER'], w, label='noisy', color='#bf8f4f')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(stt_df['spec'], rotation=30, ha='right')
axes[1].set_ylabel('DER (%)')
axes[1].set_title('DER per specjalizacja (synthetic ceiling 50-60%)')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. SOAP metrics per-spec


In [ ]:
slm_by_spec = collections.defaultdict(list)
for r in SLM['rows']:
    sid = r['scenario_id'][:8]
    scen = sid_to_scen.get(sid)
    if scen:
        slm_by_spec[scen['specialization']].append(r)

slm_df = pd.DataFrame([{
    'spec': spec,
    'schema': sum(1 for r in rs if r['schema_ok']) / len(rs) * 100,
    'dx_lex': sum(1 for r in rs if r['diag_match']) / len(rs) * 100,
    'dx_sem': sum(1 for r in rs if r['diag_match_semantic']) / len(rs) * 100,
    'ICD10': sum(1 for r in rs if r['icd10_match']) / len(rs) * 100,
    'plan': sum(1 for r in rs if r['plan_nonempty']) / len(rs) * 100,
    'cos': statistics.mean(r['diag_cos_sim'] for r in rs),
} for spec, rs in slm_by_spec.items()])
slm_df = slm_df.sort_values('spec')
slm_df

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
x = range(len(slm_df))
metrics = [('schema', '#5fbf8f'), ('dx_lex', '#bf5f5f'), ('dx_sem', '#4f8fbf'), ('ICD10', '#bf8f4f'), ('plan', '#8f5fbf')]
n = len(metrics)
w = 0.8 / n
for i, (col, color) in enumerate(metrics):
    ax.bar([j + (i - n/2 + 0.5) * w for j in x], slm_df[col], w, label=col, color=color)
ax.set_xticks(list(x))
ax.set_xticklabels(slm_df['spec'], rotation=30, ha='right')
ax.set_ylabel('%')
ax.set_title('SOAP metryki per specjalizacja (lex vs sem dx różnica = parafrazy)')
ax.legend(loc='lower right', ncol=5)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Cloud agent per-spec


In [ ]:
agent_by_spec = collections.defaultdict(list)
for r in AGENT['rows']:
    sid = r['scenario_id'][:8]
    scen = sid_to_scen.get(sid)
    if scen:
        agent_by_spec[scen['specialization']].append(r)

agent_df = pd.DataFrame([{
    'spec': spec,
    'recall': statistics.mean(r['drug_recall'] for r in rs) * 100,
    'precision': statistics.mean(r['drug_precision'] for r in rs) * 100,
    'nfz_acc': sum(1 for r in rs if r['nfz_accuracy']) / len(rs) * 100,
    'empty_ok': sum(1 for r in rs if r['empty_list_correct']) / len(rs) * 100,
    'mean_dur': statistics.mean(r['duration_s'] for r in rs),
} for spec, rs in agent_by_spec.items()])
agent_df = agent_df.sort_values('spec')
agent_df

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
x = range(len(agent_df))
metrics = [('recall', '#5fbf8f'), ('precision', '#bf5f5f'), ('nfz_acc', '#4f8fbf'), ('empty_ok', '#bf8f4f')]
n = len(metrics)
w = 0.8 / n
for i, (col, color) in enumerate(metrics):
    ax.bar([j + (i - n/2 + 0.5) * w for j in x], agent_df[col], w, label=col, color=color)
ax.set_xticks(list(x))
ax.set_xticklabels(agent_df['spec'], rotation=30, ha='right')
ax.set_ylabel('%')
ax.set_title('Cloud agent metryki per specjalizacja')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Lexical vs Semantic Diagnosis Match — central thesis claim


In [ ]:
# Cosine similarity distribution for ALL scenarios
cos_sims = [r['diag_cos_sim'] for r in SLM['rows'] if r['schema_ok']]
diag_lex_hits = sum(1 for r in SLM['rows'] if r['diag_match'])
diag_sem_hits = sum(1 for r in SLM['rows'] if r['diag_match_semantic'])

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(cos_sims, bins=20, color='#4f8fbf', edgecolor='white')
ax.axvline(0.85, color='red', linestyle='--', label='threshold (0.85)')
ax.set_xlabel('cosine similarity (predicted vs expected diagnosis)')
ax.set_ylabel('# scenariuszy')
ax.set_title(f'mmlw cosine similarity distribution\nLexical match: {diag_lex_hits}/{len(SLM["rows"])} | Semantic match (cos≥0.85): {diag_sem_hits}/{len(SLM["rows"])}')
ax.legend()
plt.tight_layout()
plt.show()

## 6. E2E latency budget


In [ ]:
stages = ['STT+diariz', 'SOAP (Bielik)', 'Anonymize', 'Cloud agent']
cold_s = [60, 120, 3, 30]
warm_s = [30, 80, 0.05, 22]

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(stages))
w = 0.35
ax.bar([i - w/2 for i in x], cold_s, w, label='cold start', color='#bf8f4f')
ax.bar([i + w/2 for i in x], warm_s, w, label='warm', color='#5fbf8f')
ax.set_xticks(list(x))
ax.set_xticklabels(stages)
ax.set_ylabel('seconds')
ax.set_title(f'E2E latency budget — cold {sum(cold_s)}s, warm {sum(warm_s):.1f}s\n(Phase 3b live-verified: 167.3s cold / 0.31s warm with 3-layer cache)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()